# 16. Month-over-month change when a month is missing

**Track:** sql | **Difficulty:** medium | **Tags:** lag, month over month, window functions, gaps, time series

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("016-month-over-month-gaps")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "monthly_revenue": """
UEFSMRUEFYYBFYwBTBUMFQASAABD8EIIAAAAU3RhbmRhcmQKAAAARW50ZXJwcmlzZQUAAABCYXNpYwcAAABQcmVtaXVtBwAAAFN0
dWRlbnQGAAAARmFtaWx5FQAVQhVGLBWAARUQFQYVBhw2ACgHU3R1ZGVudBgFQmFzaWMREQAAACGAAwAAAIABAQMRSKBRypgxGMRo
mUpFVVtFgJitHQiSxZoBFQQVYBVkTBUYFQASAAAwvBBPAADBTQAAPE4AAJhOAABaTgAAHU4AAHlOAADTTgAA4E0AAPFOAAC0TgAA
/00AABUAFVIVViwVgAEVEBUGFQYcGAQQTwAAGATBTQAAFgAoBBBPAAAYBMFNAAAREQAAACmgAwAAAIABAQQREDJUBmd3hYCXZjKq
OWmxWhp0giBbk0eRsLqBRYUyC5QVBBWABhWKBkwVgAEVABIAAIAD9H8BAAAB0s3CAAACF7YwAAAByHrKAAABqbmMAAABvsuwAAAC
3q1YAAAC1ZKuAAACOB3kAAACJAucAAACWvIaAAADCJ1IAAACaIzqAAACFMNOAAACfSJAAAAC8hv0AAAB5AMIAAABnSWgAAADbvH2
AAABlKeaAAAB04rSAAAC2VUuAAACMwi4AAAB60OsAAADQut0AAACiyYkAAAC/kCMAAACV8CGAAACO1XqAAACXA90AAACD5mgAAAC
i5nEAAACLX/sAAACias8AAAB/QAaAAACAgseAAACnBNYAAACJQDsAAAB3EaQAAACRryqAAAB37AAAAABEnKYAAABL2AQAAABsXuu
AAAC3crIAAACPWTeAAACsx5eAAACim1gAAAC6JWsAAACoLpcAAACswJwAAABcLGMAAACZ7qQAAABJJSWAAACUsyOAAACnflIAAAD
MCEaAAACYgC+AAABApiWAAAC+dKQAAACmStmAAAB9sgcAAADPXfAAAABYt6uAAAB2trkFQAVchV2LBWAARUQFQYVBhwYBgAAA27x
9hgGAAABApiWFgAoBgAAA27x9hgGAAABApiWEREAAAA54AMAAACAAQEGEUAgDERhHEiiLEzjPFAkTVRlXVimbVznfWAojmRpnmiq
rmzrvnAsz3Rt33iu73zv/xUEGUw1ABgGc2NoZW1hFQYAFQwlAhgJcGxhbl9uYW1lJQBMHAAAABUCJQIYDXJldmVudWVfbW9udGgl
DExsAAAAFQ4VDBUCGAdyZXZlbnVlJQoVBBUYLFwVBBUYAAAAFoABGRwZPCYAHBUMGTUABhAZGAlwbGFuX25hbWUVAhaAARa4AhbC
Aia0ASYIHDYAKAdTdHVkZW50GAVCYXNpYxERABksFQQVABUCABUAFRAVAgA8FpYHGQYZJgCAAQAAACYAHBUCGTUABhAZGA1yZXZl
bnVlX21vbnRoFQIWgAEWrgIWtgImygMmygIcGAQQTwAAGATBTQAAFgAoBBBPAAAYBMFNAAAREQAZLBUEFQAVAgAVABUQFQIAPCkG
GSYAgAEAAAAmABwVDhk1AAYQGRgHcmV2ZW51ZRUCFoABFoQIFpIIJqwLJoAFHBgGAAADbvH2GAYAAAECmJYWACgGAAADbvH2GAYA
AAECmJYREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAgAEAAAAW6gwWgAEmCBaKDQAZHBgMQVJST1c6c2NoZW1hGMwCLy8vLy8vQUFB
QUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQU1BQUFDTUFBQUFRQUFB
QUFRQUFBQ1EvLy8vQUFBQkJ4QUFBQUFnQUFBQUJBQUFBQUFBQUFBSEFBQUFjbVYyWlc1MVpRQUlBQXdBQkFBSUFBZ0FBQUFNQUFB
QUFnQUFBTWovLy84QUFBRUlFQUFBQUNRQUFBQUVBQUFBQUFBQUFBMEFBQUJ5WlhabGJuVmxYMjF2Ym5Sb0FBWUFDQUFHQUFZQUFB
QUFBQUFBRUFBVUFBZ0FCZ0FIQUF3QUFBQVFBQkFBQUFBQUFBRUZFQUFBQUNBQUFBQUVBQUFBQUFBQUFBa0FBQUJ3YkdGdVgyNWhi
V1VBQUFBRUFBUUFCQUFBQUFBQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRk8HAAAHAAAHAAAACQDAABQ
QVIx
""",
}

_data_dir = pathlib.Path("data") / "016-month-over-month-gaps"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

monthly_revenue = spark.read.parquet((_data_dir / "monthly_revenue.parquet").resolve().as_posix())
monthly_revenue.createOrReplaceTempView("monthly_revenue")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'plan_name': '3084451aefd8a54b31657d7b4acf35ac4415f5aa572db80acc44ecfb0276069a',
                         'prev_month_revenue': '2cc23c354bd20a3840022c3b07d2fc7f19e885b108f6be1b0df7118ceba51410',
                         'revenue': '830041662f788cb63616a55036f91f0a3d536297b0f19fffd551755b9e8e641f',
                         'revenue_change': '276b1dc94f9756dff7e547a0189d57039e96275e1a746fdfe9c46344b4099c00',
                         'revenue_month': 'ab872167b93cd186e16fc80473b49c1b20d38665eaa2bd12dac4429d9c45f8dd'},
 'columns': [['plan_name', 'string'],
             ['prev_month_revenue', 'decimal(12,2)'],
             ['revenue', 'decimal(12,2)'],
             ['revenue_change', 'decimal(13,2)'],
             ['revenue_month', 'date']],
 'fingerprint': '27ffa536ef7d70b481f19b2aa0e67dce2564498a7bf8270c86ece8fdffbbdd4a',
 'order_matters': False,
 'row_count': 64}

check = make_check(EXPECTED)

## The situation

Finance tracks how each subscription plan's revenue changes from one month to the
next. The source table only has a row for a plan in a month when the plan earned
something.

The current report shows the Standard plan jumping sharply in December. It did not.
The report compared December with October, because November has no row.

## Input tables

### `monthly_revenue`

One row per plan and month in which the plan had revenue. Months with no revenue
have no row.

| Column | Type | Description |
|---|---|---|
| `plan_name` | STRING | The subscription plan. |
| `revenue_month` | DATE | The first day of the month. |
| `revenue` | DECIMAL(12,2) | Revenue of the plan in that month. |

## Requirement

Return every row of `monthly_revenue` with the previous month's revenue next to it.

- `prev_month_revenue` is the plan's revenue in the calendar month immediately
  before `revenue_month`. January 2025 is compared with December 2024.
- If the plan has no row for that previous month, `prev_month_revenue` is NULL.
  Never compare with a month further back.
- `revenue_change` is `revenue` minus `prev_month_revenue`, and NULL when
  `prev_month_revenue` is NULL.

## Expected output

One row per row of `monthly_revenue`.

| Column | Type | Description |
|---|---|---|
| `plan_name` | STRING | |
| `revenue_month` | DATE | |
| `revenue` | DECIMAL(12,2) | |
| `prev_month_revenue` | DECIMAL(12,2) | NULL if the plan has no row for the previous month. |
| `revenue_change` | DECIMAL(13,2) | NULL if `prev_month_revenue` is NULL. |

In [ ]:
monthly_revenue.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `monthly_revenue`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)